In [1]:
import sqlite3
## Creating a database

def get_connection():
 conn = sqlite3.connect("tax_arrears.db")
 conn.execute("PRAGMA foreign_keys = ON")  # Enforces foreign key constraints
 return conn
 
conn = get_connection()
cursor = conn.cursor()


In [3]:
import sqlite3
## Creating a database

def get_connection():
 conn = sqlite3.connect("tax_arrears1.db")
 conn.execute("PRAGMA foreign_keys = ON")  # Enforces foreign key constraints
 return conn
 
conn = get_connection()
cursor = conn.cursor()

# Creating Tables

In [4]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS taxpayers (
        pin TEXT PRIMARY KEY,
        name TEXT
    )
    """)

cursor.execute("""
    CREATE TABLE IF NOT EXISTS tax_records (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pin TEXT NOT NULL,
    station TEXT NOT NULL,
    phone_number TEXT NOT NULL,
    obligation TEXT NOT NULL,
    from_date DATE NOT NULL,
    to_date DATE NOT NULL,
    principal REAL NOT NULL CHECK (principal >= 0),
    penalty REAL NOT NULL CHECK (penalty >= 0),
    interest REAL NOT NULL CHECK (interest >= 0),
    status TEXT DEFAULT 'UNPAID',

    FOREIGN KEY (pin)
        REFERENCES taxpayers(pin)
        ON DELETE CASCADE
    )
    """)

cursor.execute("""
    CREATE TABLE IF NOT EXISTS payments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        pin TEXT,
        amount REAL,
        payment_date TEXT,
        FOREIGN KEY(pin) REFERENCES taxpayers(pin)
    )
    """)

conn.commit()
conn.close()

# Creating Classes

In [ ]:
from datetime import date, datetime

class Taxpayer:
    def __init__(self, pin, name):
        self.pin = pin
        self.name = name

    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR IGNORE INTO taxpayers (pin, name) VALUES (?, ?)",
            (self.pin, self.name)
        )
        conn.commit()
        conn.close()


class TaxRecord:
    def __init__(self,id, pin,station,phone_number, obligation, from_date, to_date, principal, penalty, interest):
        self.id = id
        self.pin = pin
        self.station = station
        self.phone_number = phone_number
        self.obligation = obligation
        self.from_date = from_date
        self.to_date = to_date
        self.principal = principal
        self.penalty = penalty
        self.interest = interest
    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO tax_records (id, pin, station, phone_number, obligation, from_date, to_date, principal, penalty, interest)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (self.id, self.pin, self.station, self.phone_number, self.obligation, self.from_date, self.to_date, self.principal, self.penalty, self.interest))
        conn.commit()
        conn.close()


class Payment:
    def __init__(self,id, pin, amount, payment_date):
        self.id = id
        self.pin = pin
        self.amount = amount
        self.payment_date = payment_date

    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO payments (id, pin, amount, payment_date)
            VALUES (?, ?, ?, ?)
        """, (self.id, self.pin, self.amount, self.payment_date))
        conn.commit()
        conn.close()


class ArrearsAnalyzer:
    @staticmethod
    def arrears_age(assessment_date):
        days = (date.today() - datetime.fromisoformat(assessment_date).date()).days

        if days <= 180:
            return "0–6 Months", "Low"
        elif days <= 365:
            return "6–12 Months", "Medium"
        else:
            return "Over 12 Months", "High"


In [ ]:

def register_taxpayer():
    pin = input("Enter PIN: ")
    name = input("Enter Name: ")
    Taxpayer(pin, name).save()
    print("Taxpayer registered successfully.\n")

def add_tax_records():
    id = int(input("Enter Tax Record ID: "))
    pin = input("Enter Taxpayer PIN: ")
    station = input("Enter Tax Station: ")
    phone_number = input("Enter Phone Number: ")
    obligation = input("Enter Tax Type (VAT, PAYE, IT): ")
    from_date = input("Enter From Date (YYYY-MM-DD): ")
    to_date = input("Enter To Date (YYYY-MM-DD): ")
    principal = float(input("Enter Principal Amount: "))
    penalty = float(input("Enter Penalty Amount: "))
    interest = float(input("Enter Interest Amount: "))
   
    TaxRecord(id, pin, station, phone_number, obligation, from_date, to_date, principal, penalty, interest).save()
    print("Tax assessment added.\n")

def record_payment():
    id = int(input("Enter Tax Record ID: "))
    pin = input("Enter Taxpayer PIN: ")
    amount = float(input("Enter Payment Amount: "))
    payment_date = date.today().isoformat()
    Payment(id, pin, amount, payment_date).save()
    print("Payment recorded.\n")

def view_arrears():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    SELECT tr.id, t.pin, t.name, tr.station, tr.phone_number, tr.obligation,
           tr.principal + tr.penalty + tr.interest,
           IFNULL(SUM(p.amount), 0),
           tr.assessment_date
    FROM tax_records tr
    JOIN taxpayers t ON tr.pin = t.pin
    LEFT JOIN payments p ON tr.id = p.tax_record_id
    GROUP BY tr.id
    """)

    rows = cursor.fetchall()
    conn.close()

    for r in rows:
        balance = r[4] - r[5]
        age, risk = ArrearsAnalyzer.arrears_age(r[6])

        print(f"""
Tax Record ID: {r[0]}
PIN: {r[1]}
Name: {r[2]}
Tax Type: {r[3]}
Outstanding Balance: {balance}
Arrears Age: {age}
Risk Level: {risk}
        """)

def menu():
    while True:
        print("""
TAX ARREARS MANAGEMENT SYSTEM
1. Register Taxpayer
2. Add Tax Assessment
3. Record Payment
4. View Arrears Report
5. Exit
        """)

        choice = input("Choose an option: ")

        if choice == "1":
            register_taxpayer()
        elif choice == "2":
            add_tax_record()
        elif choice == "3":
            record_payment()
        elif choice == "4":
            view_arrears()
        elif choice == "5":
            print("Exiting system...")
            break
        else:
            print("Invalid choice.\n")

menu()



TAX ARREARS MANAGEMENT SYSTEM
1. Register Taxpayer
2. Add Tax Assessment
3. Record Payment
4. View Arrears Report
5. Exit
        


TypeError: TaxRecord.__init__() takes 5 positional arguments but 11 were given

In [23]:
import pandas as pd

df = pd.read_csv("Taxpayer Data.csv")

for _, row in df.iterrows():
    cursor.execute(
        "INSERT INTO taxpayers VALUES (?, ?)",
        (row["pin"], row["name"])
    )

FileNotFoundError: [Errno 2] No such file or directory: 'Taxpayer Data.csv'